# Diagnostic Qwen3.6-27B-FP8 — Domino / H100 (v2)

**Objectif** : isoler la cause des sorties incohérentes (tokens chinois/garbage).  
**Hypothèse principale** : le binaire du kernel FP8 (`finegrained_fp8`) est incompatible avec l'environnement Domino → il *charge* mais *calcule faux*.  

## Structure du diagnostic
| Étape | Ce qu'on teste | Critère de passage |
|-------|----------------|--------------------|
| A | Kernel FP8 — inspection binaire + validation numérique | Max err < 5% sur matmul de référence |
| B | Chat template — inspection des tokens réels | Format `<\|im_start\|>` présent, `<think>` absent |
| C | **Modèle en BF16** — contourne totalement FP8 | Génère "OK" ou texte cohérent |
| D | Modèle en FP8 (si C passe) | Même résultat qu'en BF16 |
| E | Test vision sur une page PDF (si D passe) | JSON extrait correctement |

> **Règle d'or** : ne pas passer à l'étape suivante si la précédente échoue.

In [ ]:
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
# CELLULE 1 — Variables d'environnement (AVANT tout import transformers)
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
import os

os.environ["HF_HUB_OFFLINE"]                   = "1"
os.environ["TRANSFORMERS_OFFLINE"]              = "1"
os.environ["USE_HUB_KERNELS"]                  = "0"
# Désactive DeepGEMM (nvcc absent sur Domino) — évite le message de fallback
os.environ["TRANSFORMERS_DISABLE_DEEPGEMM_LINEAR"] = "1"

for k, v in os.environ.items():
    if any(kw in k for kw in ["HF_", "TRANSFORMERS_", "USE_HUB"]):
        print(f"  {k} = {v}")

In [ ]:
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
# CELLULE 2 — Environnement complet
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
import sys, subprocess
import torch
import transformers

print("Python       :", sys.version.split()[0])
print("Torch        :", torch.__version__)
print("CUDA Torch   :", torch.version.cuda)
print("Transformers :", transformers.__version__)
print("CUDA dispo   :", torch.cuda.is_available())

if torch.cuda.is_available():
    props = torch.cuda.get_device_properties(0)
    print(f"GPU          : {props.name}")
    print(f"VRAM totale  : {props.total_memory / 1024**3:.1f} GB")
    print(f"Compute cap  : {props.major}.{props.minor}")
    # sm_90 = H100 ; vérifier que le kernel FP8 a bien été compilé pour sm_90
    print(f"  → sm_{props.major}{props.minor} ({'Hopper ✓' if props.major == 9 else 'Non-Hopper ⚠️'})")

# nvcc disponible ?
nvcc = subprocess.run(["which", "nvcc"], capture_output=True, text=True)
print(f"nvcc         : {'ABSENT ⚠️' if nvcc.returncode != 0 else nvcc.stdout.strip()}")

# CUDA_HOME
print(f"CUDA_HOME    : {os.environ.get('CUDA_HOME', 'non défini')}")

In [ ]:
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
# CELLULE 3 — Chemin du modèle
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
from pathlib import Path

MODEL_PATH = Path("/domino/edv/modelhub/ModelHub-model-huggingface-Qwen/Qwen3.6-27B-FP8/main")
print("MODEL_PATH :", MODEL_PATH)
print("Existe     :", MODEL_PATH.exists())

required = [
    "config.json", "generation_config.json", "preprocessor_config.json",
    "tokenizer_config.json", "tokenizer.json", "chat_template.jinja",
    "model.safetensors.index.json"
]
all_ok = True
for f in required:
    ok = (MODEL_PATH / f).exists()
    print(f"  {f:40} : {'✓' if ok else '✗ MANQUANT'}")
    if not ok:
        all_ok = False

# Afficher la quantization_config du modèle (clé pour comprendre le type FP8)
import json
cfg = json.loads((MODEL_PATH / "config.json").read_text())
print("\n--- quantization_config ---")
print(json.dumps(cfg.get("quantization_config", "ABSENTE"), indent=2))
print("model_type :", cfg.get("model_type"))
print("torch_dtype:", cfg.get("torch_dtype"))

assert all_ok, "Fichiers manquants dans MODEL_PATH"

---
## ÉTAPE A — Diagnostic du kernel FP8

**Hypothèse** : `import finegrained_fp8` réussit mais le binaire `.so` a été compilé  
pour une version CUDA/PyTorch différente de l'environnement Domino.  
Les calculs FP8 produisent alors des résultats numériquement incorrects → tokens garbage.

Deux sous-tests :
1. **Inspection binaire** — quelles architectures GPU et versions sont dans le `.so` ?
2. **Validation numérique** — le résultat d'un matmul connu est-il correct ?

In [ ]:
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
# CELLULE A1 — Inspection du binaire du kernel FP8
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
import subprocess, json
from pathlib import Path

KERNEL_DIR = Path("/mnt/finegrained-fp8")
BUILD_DIR  = KERNEL_DIR / "build" / "torch-cuda"

# 1. Métadonnées
meta_path = KERNEL_DIR / "metadata.json"
if meta_path.exists():
    meta = json.loads(meta_path.read_text())
    print("=== metadata.json ===")
    print(json.dumps(meta, indent=2))
else:
    print("⚠️  metadata.json introuvable")

# 2. Fichiers .so présents
so_files = list(BUILD_DIR.glob("*.so"))
print(f"\n=== Fichiers .so dans {BUILD_DIR} ===")
for s in so_files:
    print(f"  {s.name}  ({s.stat().st_size/1024:.0f} KB)")

# 3. Strings du binaire : extraire les mentions cuda/sm_/torch
if so_files:
    print("\n=== Strings de build (CUDA/arch/torch) dans le .so ===")
    res = subprocess.run(
        ["strings", str(so_files[0])],
        capture_output=True, text=True, timeout=30
    )
    hits = [
        l for l in res.stdout.splitlines()
        if any(kw in l.lower() for kw in ["cuda", "sm_", "ptxas", "arch=", "torch", "compute_"])
        and len(l) > 3
    ]
    if hits:
        for h in hits[:40]:   # limiter l'affichage
            print(" ", h)
    else:
        print("  (aucune chaîne cuda/sm détectée — binaire peut-être stripé)")
    
    # Essayer cuobjdump si disponible
    cmd = subprocess.run(["which", "cuobjdump"], capture_output=True, text=True)
    if cmd.returncode == 0:
        print("\n=== cuobjdump --list-elf (architectures compilées) ===")
        res2 = subprocess.run(
            ["cuobjdump", "--list-elf", str(so_files[0])],
            capture_output=True, text=True, timeout=30
        )
        print(res2.stdout or "  (aucune sortie)")
    else:
        print("\ncuobjdump : absent (normal sur Domino)")
else:
    print("⚠️  Aucun .so trouvé dans", BUILD_DIR)

In [ ]:
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
# CELLULE A2 — Validation NUMÉRIQUE du kernel FP8
# Principe : A @ B = C doit être égal à la référence BF16
#            Si l'erreur est > 5%, le kernel calcule FAUX
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
import sys, inspect
import torch

BUILD_DIR_STR = "/mnt/finegrained-fp8/build/torch-cuda"
if BUILD_DIR_STR not in sys.path:
    sys.path.insert(0, BUILD_DIR_STR)

try:
    import finegrained_fp8
    print("Import finegrained_fp8 : OK")
    print("Module file :", getattr(finegrained_fp8, '__file__', '?'))
    funcs = [x for x in dir(finegrained_fp8) if not x.startswith('_')]
    print("Fonctions   :", funcs)
except ImportError as e:
    print("❌ Import échoué :", e)
    raise

device = torch.device('cuda:0')
N = 128   # matrice NxN

# Référence en float32 : A = 2*I, B = 3*I  →  A@B = 6*I
A_f32 = torch.diag(torch.full((N,), 2.0, device=device))
B_f32 = torch.diag(torch.full((N,), 3.0, device=device))
ref_f32 = A_f32 @ B_f32  # résultat attendu : 6.0 sur la diagonale

# Conversion FP8
A_fp8 = A_f32.to(torch.float8_e4m3fn)
B_fp8 = B_f32.to(torch.float8_e4m3fn)
# Note : B est transposé car la plupart des GEMM FP8 attendent B^T
B_fp8_t = B_f32.T.contiguous().to(torch.float8_e4m3fn)

# Scales de quantification (1.0 = pas de mise à l'échelle)
scale_a = torch.tensor(1.0, device=device)
scale_b = torch.tensor(1.0, device=device)

print(f"\nTest matmul {N}×{N} (valeur attendue sur diagonale : 6.0)")
print("Ref BF16 diag[0,0] :", ref_f32[0, 0].item())

# ─── Tentative 1 : signature standard (A, B^T, scale_a, scale_b) ───
for b_arg, b_label in [(B_fp8_t, "B^T"), (B_fp8, "B")]:
    for sig in [
        lambda b=b_arg: finegrained_fp8.matmul(A_fp8, b, scale_a, scale_b),
        lambda b=b_arg: finegrained_fp8.matmul(A_fp8, b, scale_a, scale_b, None),
        lambda b=b_arg: finegrained_fp8.matmul(A_fp8, b, scale_a.unsqueeze(0), scale_b.unsqueeze(0)),
    ]:
        try:
            out = sig()
            torch.cuda.synchronize()
            out_f32 = out.to(torch.float32)
            diag_val = out_f32[0, 0].item()
            max_err  = (out_f32 - ref_f32).abs().max().item()
            rel_err  = max_err / 6.0
            print(f"\n  Output dtype : {out.dtype}, shape : {out.shape}")
            print(f"  Diag[0,0]    : {diag_val:.4f}  (attendu 6.0)")
            print(f"  Max err abs  : {max_err:.6f}")
            print(f"  Max err rel  : {rel_err*100:.2f}%")
            if rel_err < 0.05:
                print("  ✅ Kernel FP8 NUMÉRIQUEMENT CORRECT")
            else:
                print("  ❌ Kernel FP8 INCORRECT — binaire probablement incompatible avec CUDA/Torch courant")
                print("     → Passer directement à l'ÉTAPE C (BF16 bypass)")
            break
        except Exception as exc:
            print(f"  Signature {b_label} tentée → {type(exc).__name__}: {exc}")
            continue
    else:
        continue
    break
else:
    print("\n⚠️  Toutes les signatures ont échoué — impossible de valider numériquement")
    print("   Cela ne signifie pas que le kernel est correct : il faudra passer à l'ÉTAPE C")

---
## ÉTAPE B — Inspection du Chat Template

Vérifie que les tokens réels envoyés au modèle ont le bon format Qwen3.  
**Ce test ne charge pas encore le modèle** — juste le processor/tokenizer.

In [ ]:
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
# CELLULE B1 — Chargement du processor + inspection du template
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
from transformers import AutoProcessor
import torch

processor = AutoProcessor.from_pretrained(
    str(MODEL_PATH),
    local_files_only=True,
    trust_remote_code=True
)
print("Processor type :", type(processor).__name__)
print("Tokenizer  type:", type(processor.tokenizer).__name__)

# Tokens spéciaux Qwen3
tok = processor.tokenizer
special_ids = {
    "im_start" : tok.convert_tokens_to_ids("<|im_start|>"),
    "im_end"   : tok.convert_tokens_to_ids("<|im_end|>"),
    "think"    : tok.convert_tokens_to_ids("<think>"),
    "/think"   : tok.convert_tokens_to_ids("</think>"),
    "EOS"      : tok.eos_token_id,
    "BOS"      : tok.bos_token_id,
    "PAD"      : tok.pad_token_id,
}
print("\nTokens spéciaux :")
for name, tid in special_ids.items():
    print(f"  {name:12} : {tid}")

# Construire un prompt minimal et inspecter les token IDs
messages_test = [{
    "role": "user",
    "content": [{"type": "text", "text": "Réponds uniquement par OK."}]
}]

inputs = processor.apply_chat_template(
    messages_test,
    add_generation_prompt=True,
    tokenize=True,
    return_dict=True,
    return_tensors="pt",
    enable_thinking=False
)

ids   = inputs["input_ids"][0].tolist()
text  = tok.decode(ids)
print(f"\nNombre de tokens : {len(ids)}")
print("Texte décodé     :\n", repr(text))
print("\nToken IDs bruts  :", ids)

# Contrôles
print("\nContrôles :")
print("  <|im_start|> présent :", special_ids["im_start"] in ids)
print("  <|im_end|>   présent :", special_ids["im_end"]   in ids)
think_id = special_ids.get("think")
if think_id and think_id != 0:
    print("  <think>      absent  :", think_id not in ids, "← doit être True avec enable_thinking=False")
print("  Dernier token = assistant header :",
      tok.decode([ids[-1]]) if ids else "?",
      "(doit être vide ou marquer le début de la réponse)")

---
## ÉTAPE C — Test BF16 (contourne entièrement FP8)

Chargement du modèle en BF16 pour établir une **baseline saine**.  
Si la génération est correcte ici mais incorrecte en FP8 (Étape D) → **le kernel FP8 est la cause**.  

> ℹ️ Avec `torch_dtype=torch.bfloat16`, Transformers dequantise les poids FP8 à la volée.  
> VRAM requise : ~55 GB pour 27B en BF16 (margin serrée sur H100 80 GB, surveiller).

In [ ]:
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
# CELLULE C1 — Chargement modèle en BF16
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
import gc
import torch
from transformers import AutoModelForMultimodalLM

# Libérer la mémoire GPU avant le chargement
if 'model_fp8' in dir():
    del model_fp8
    gc.collect()
    torch.cuda.empty_cache()

print("VRAM libre avant chargement :",
      round((torch.cuda.get_device_properties(0).total_memory -
             torch.cuda.memory_allocated()) / 1024**3, 1), "GB")

print("\nChargement Qwen3.6-27B en BF16 (dequantise depuis FP8)...")
print("Patience : ~2-3 min selon la vitesse du stockage Domino")

try:
    model_bf16 = AutoModelForMultimodalLM.from_pretrained(
        str(MODEL_PATH),
        local_files_only=True,
        trust_remote_code=True,
        device_map="auto",
        torch_dtype=torch.bfloat16,   # ← force BF16, bypass FP8 kernel
    )
    model_bf16.eval()
    print("\n✅ MODEL BF16 OK")
    print("Type   :", type(model_bf16).__name__)
    print("Dtype  :", next(model_bf16.parameters()).dtype)
    print("VRAM   :", round(torch.cuda.memory_allocated() / 1024**3, 1), "GB allouées")

except RuntimeError as e:
    # VRAM insuffisante pour 27B en BF16 → essayer avec load_in_8bit
    print("❌ RuntimeError BF16 :", e)
    print("\n→ Tentative avec quantization_config override (off)...")
    # Parfois 'torch_dtype' seul ne suffit pas — il faut aussi supprimer la quant config
    import copy
    cfg_override = copy.deepcopy(cfg)  # cfg chargé en cellule 3
    cfg_override.pop("quantization_config", None)
    print("  Conseil : modifier temporairement config.json en retirant 'quantization_config'")
    print("  puis relancer avec torch_dtype=torch.bfloat16")
    raise

In [ ]:
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
# CELLULE C2 — Génération TEXTE en BF16 (test minimal)
# Attendu : réponse courte et cohérente ("OK" ou équivalent)
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
import torch

messages_simple = [{
    "role": "user",
    "content": [{"type": "text", "text": "Réponds uniquement par OK."}]
}]

inputs = processor.apply_chat_template(
    messages_simple,
    add_generation_prompt=True,
    tokenize=True,
    return_dict=True,
    return_tensors="pt",
    enable_thinking=False
)
inputs = {k: v.to(model_bf16.device) if torch.is_tensor(v) else v
          for k, v in inputs.items()}

print(f"Tokens IN : {inputs['input_ids'].shape[1]}")

with torch.inference_mode():
    output_ids = model_bf16.generate(
        **inputs,
        max_new_tokens=20,
        do_sample=False,
        temperature=None,
        top_p=None,
    )

new_ids = output_ids[:, inputs["input_ids"].shape[1]:]
result  = processor.batch_decode(
    new_ids, skip_special_tokens=True, clean_up_tokenization_spaces=False
)
raw_ids = new_ids[0].tolist()

print(f"Tokens OUT: {len(raw_ids)}")
print(f"IDs bruts : {raw_ids}")
print(f"\n{'='*40}")
print(f"RÉSULTAT BF16 : {result}")
print(f"{'='*40}")

# Diagnostic automatique
res_text = result[0] if result else ""
if any(ord(c) > 0x4E00 for c in res_text):
    print("⚠️  Caractères CJK détectés — sortie possiblement incohérente même en BF16")
    print("   Cela indiquerait un problème de chat template ou de classe de modèle")
elif res_text.strip() == "":
    print("⚠️  Sortie vide — vérifier les special tokens / EOS prématuré")
else:
    print("✅ Sortie BF16 cohérente → le chat template et le modèle fonctionnent")
    print("   Si FP8 (Étape D) produit du garbage → kernel FP8 confirmé fautif")

In [ ]:
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
# CELLULE C3 — Test question ouverte (BF16)
# Valide que la cohérence n'est pas juste sur un token
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
messages_open = [{
    "role": "user",
    "content": [{"type": "text", "text": "Quelle est la capitale de l'Algérie ? Réponds en une phrase."}]
}]

inputs2 = processor.apply_chat_template(
    messages_open,
    add_generation_prompt=True,
    tokenize=True,
    return_dict=True,
    return_tensors="pt",
    enable_thinking=False
)
inputs2 = {k: v.to(model_bf16.device) if torch.is_tensor(v) else v
           for k, v in inputs2.items()}

with torch.inference_mode():
    out2 = model_bf16.generate(**inputs2, max_new_tokens=60, do_sample=False)

new2 = out2[:, inputs2["input_ids"].shape[1]:]
print("RÉSULTAT BF16 (question ouverte) :")
print(processor.batch_decode(new2, skip_special_tokens=True)[0])

---
## ÉTAPE D — Test FP8 (ne continuer que si BF16 fonctionne)

Charge le modèle avec `dtype="auto"` (poids en FP8, kernel activé).  
Compare le résultat au BF16 de l'Étape C.

In [ ]:
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
# CELLULE D1 — Patch finegrained_fp8 + chargement FP8
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
import gc, sys, torch
from transformers import AutoModelForMultimodalLM
import transformers

# Libérer le modèle BF16
if 'model_bf16' in dir():
    del model_bf16
    gc.collect()
    torch.cuda.empty_cache()
print("VRAM libre :", round((torch.cuda.get_device_properties(0).total_memory
                              - torch.cuda.memory_allocated()) / 1024**3, 1), "GB")

# ─── Patch : forcer l'utilisation du kernel local ───
BUILD_DIR_STR = "/mnt/finegrained-fp8/build/torch-cuda"
if BUILD_DIR_STR not in sys.path:
    sys.path.insert(0, BUILD_DIR_STR)

import finegrained_fp8 as _local_fp8

try:
    import transformers.integrations.finegrained_fp8 as _tfp8
    _tfp8.FineGrainedFP8 = _local_fp8
    print("Patch transformers.integrations.finegrained_fp8 : OK")
except Exception as e:
    print("⚠️  Patch échoué :", e)
    print("   Vérifier le chemin d'import dans la version Transformers courante")

# ─── Chargement FP8 ───
print("\nChargement Qwen3.6-27B FP8 (dtype=auto)...")
model_fp8 = AutoModelForMultimodalLM.from_pretrained(
    str(MODEL_PATH),
    local_files_only=True,
    trust_remote_code=True,
    device_map="auto",
    dtype="auto",
)
model_fp8.eval()

print("MODEL FP8 OK")
print("Dtype  :", next(model_fp8.parameters()).dtype)
print("VRAM   :", round(torch.cuda.memory_allocated() / 1024**3, 1), "GB")

In [ ]:
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
# CELLULE D2 — Génération TEXTE en FP8
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
messages_simple = [{
    "role": "user",
    "content": [{"type": "text", "text": "Réponds uniquement par OK."}]
}]

inputs = processor.apply_chat_template(
    messages_simple,
    add_generation_prompt=True,
    tokenize=True,
    return_dict=True,
    return_tensors="pt",
    enable_thinking=False
)
inputs = {k: v.to(model_fp8.device) if torch.is_tensor(v) else v
          for k, v in inputs.items()}

with torch.inference_mode():
    out_fp8 = model_fp8.generate(
        **inputs, max_new_tokens=20, do_sample=False,
        temperature=None, top_p=None
    )

new_fp8 = out_fp8[:, inputs["input_ids"].shape[1]:]
result_fp8 = processor.batch_decode(new_fp8, skip_special_tokens=True)[0]
print("RÉSULTAT FP8 :", result_fp8)

if any(ord(c) > 0x4E00 for c in result_fp8):
    print("❌ Garbage FP8 confirmé → kernel binaire incompatible")
    print("\n=== PLAN D'ACTION ===")
    print("Option 1 : utiliser exclusivement BF16 (Étape C) pour votre pipeline OCR")
    print("Option 2 : recompiler finegrained-fp8 depuis source dans l'environnement Domino")
    print("           (nécessite nvcc — voir note en bas du notebook)")
    print("Option 3 : désactiver le kernel FP8 maison et utiliser le fallback Triton pur")
    print("           → supprimer le patch + TRANSFORMERS_DISABLE_DEEPGEMM_LINEAR=1")
else:
    print("✅ FP8 produit une sortie cohérente — kernel OK")

---
## ÉTAPE E — Test Vision sur une page PDF

**Ne continuer que si la génération texte (BF16 ou FP8) est cohérente.**  
Modifier `PDF_TEST` ci-dessous avant d'exécuter.

In [ ]:
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
# CELLULE E1 — PDF → Image (première page)
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
import fitz, io, os
from PIL import Image
from IPython.display import display

PDF_TEST = "/mnt/data/ton_fichier_test.pdf"   # ← À MODIFIER
assert os.path.exists(PDF_TEST), f"PDF introuvable : {PDF_TEST}"

doc  = fitz.open(PDF_TEST)
page = doc[0]
pix  = page.get_pixmap(matrix=fitz.Matrix(2.0, 2.0), alpha=False)
image = Image.open(io.BytesIO(pix.tobytes("png"))).convert("RGB")

print(f"PDF : {len(doc)} pages")
print(f"Image page 1 : {image.size} px")
display(image)

In [ ]:
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
# CELLULE E2 — Test Vision (utilise le modèle qui a passé l'Étape C/D)
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
import torch

# Choisir le modèle qui fonctionne
model_active = model_bf16 if 'model_bf16' in dir() else model_fp8
print("Modèle utilisé :", type(model_active).__name__,
      "|", next(model_active.parameters()).dtype)

messages_vision = [{
    "role": "user",
    "content": [
        {"type": "image", "image": image},
        {"type": "text", "text": (
            "Analyse cette page.\n"
            "Retourne UNIQUEMENT ce JSON valide, sans markdown :\n"
            '{\n'
            '  "type_document": null,\n'
            '  "titre_detecte": null,\n'
            '  "langue": null\n'
            '}'
        )}
    ]
}]

inputs_v = processor.apply_chat_template(
    messages_vision,
    add_generation_prompt=True,
    tokenize=True,
    return_dict=True,
    return_tensors="pt",
    enable_thinking=False
)
inputs_v = {k: v.to(model_active.device) if torch.is_tensor(v) else v
            for k, v in inputs_v.items()}

print(f"Tokens IN : {inputs_v['input_ids'].shape[1]}")

with torch.inference_mode():
    out_v = model_active.generate(
        **inputs_v,
        max_new_tokens=200,
        do_sample=False,
        temperature=None,
        top_p=None,
    )

new_v  = out_v[:, inputs_v["input_ids"].shape[1]:]
result_v = processor.batch_decode(new_v, skip_special_tokens=True)[0]

print(f"\n{'='*50}")
print("RÉSULTAT VISION")
print(f"{'='*50}")
print(result_v)

# Vérification JSON
import json
try:
    parsed = json.loads(result_v.strip())
    print("\n✅ JSON valide :", parsed)
except Exception:
    print("\n⚠️  Sortie non parseable comme JSON")

In [ ]:
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
# CELLULE FINALE — État GPU
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
import torch
if torch.cuda.is_available():
    print("GPU          :", torch.cuda.get_device_name(0))
    print("VRAM allouée :", round(torch.cuda.memory_allocated()  / 1024**3, 2), "GB")
    print("VRAM réservée:", round(torch.cuda.memory_reserved()   / 1024**3, 2), "GB")
    print("VRAM libre   :", round((torch.cuda.get_device_properties(0).total_memory
                                   - torch.cuda.memory_reserved()) / 1024**3, 2), "GB")

---
## Note : Recompiler finegrained-fp8 si le kernel est incompatible

Si le test A2 révèle une erreur numérique > 5%, le binaire `.so` actuel a été compilé  
pour une version CUDA/PyTorch différente de Domino. Options :

### Option 1 — Triton pur (sans nvcc)
Supprimer le patch du kernel local. Avec `TRANSFORMERS_DISABLE_DEEPGEMM_LINEAR=1`,  
Transformers utilise une implémentation Triton-native pour FP8 sur H100.  
Triton JIT sur H100 ne nécessite pas nvcc.

```python
# Ne PAS injecter sys.path, ne PAS patcher transformers.integrations.finegrained_fp8
# Charger directement :
model = AutoModelForMultimodalLM.from_pretrained(..., dtype="auto")
# TRANSFORMERS_DISABLE_DEEPGEMM_LINEAR=1 suffit
```

### Option 2 — Recompiler depuis les sources
Si nvcc est disponible dans un autre conteneur Domino :
```bash
cd /mnt/finegrained-fp8
pip install . --no-build-isolation  # recompile pour le Torch/CUDA courant
```

### Option 3 — BF16 pour le pipeline
Charger avec `torch_dtype=torch.bfloat16`.  
Consommation VRAM ≈ 55 GB pour 27B params — tient sur H100 80 GB.  
Latence légèrement plus élevée qu'en FP8 natif, mais résultats fiables.